In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/egil10/fys5429.git"
    repo_dir = "/content/fys5429"

    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    else:
        !git -C {repo_dir} pull

    os.chdir(os.path.join(repo_dir, "code", "notebooks"))

    import shutil
    if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')

    from google.colab import drive
    drive.mount('/content/drive')

print(f"Working directory: {os.getcwd()}")

In [ ]:
torch.manual_seed(42)

# Domain
S_max = 300.0
T_max = 1.0
K = 100.0
r = 0.05

# Point counts
N_INTERIOR = 5000
N_IC = 1000
N_BC = 500

# Output
if IN_COLAB:
    out_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda")
else:
    out_dir = Path("..") / "data" / "generated"

out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "hs_collocation.parquet"

In [ ]:
def generate_collocation(S_max, T_max, K, r, N_interior, N_ic, N_bc):
    """
    Generate collocation points with clustered sampling
    near the strike and near expiry.
    """

    # --- Interior points (Latin Hypercube-style + clustering) ---
    # 60% uniform, 20% clustered near strike, 20% clustered near expiry
    n_uniform = int(0.6 * N_interior)
    n_strike = int(0.2 * N_interior)
    n_expiry = N_interior - n_uniform - n_strike

    # Uniform
    S_uni = torch.rand(n_uniform, 1) * S_max
    tau_uni = torch.rand(n_uniform, 1) * T_max

    # Clustered near strike: S ~ Normal(K, 0.2*K), clipped to [0, S_max]
    S_strike = (K + 0.2 * K * torch.randn(n_strike, 1)).clamp(0, S_max)
    tau_strike = torch.rand(n_strike, 1) * T_max

    # Clustered near expiry: tau ~ Beta(0.5, 2) * T_max (biased toward 0)
    tau_expiry = torch.distributions.Beta(0.5, 2.0).sample((n_expiry, 1)) * T_max
    S_expiry = torch.rand(n_expiry, 1) * S_max

    S_interior = torch.cat([S_uni, S_strike, S_expiry], dim=0)
    tau_interior = torch.cat([tau_uni, tau_strike, tau_expiry], dim=0)

    # --- IC points (tau=0): cluster near the strike ---
    # 50% uniform, 50% near strike
    n_ic_uni = N_ic // 2
    n_ic_strike = N_ic - n_ic_uni

    S_ic = torch.cat([
        torch.rand(n_ic_uni, 1) * S_max,
        (K + 0.15 * K * torch.randn(n_ic_strike, 1)).clamp(0, S_max)
    ], dim=0)
    tau_ic = torch.zeros(N_ic, 1)

    # --- BC points: S=0 (lower) + S=S_max (upper) ---
    n_bc_lower = N_bc // 2
    n_bc_upper = N_bc - n_bc_lower

    # Lower BC: V(0, tau) = 0
    S_bc_lower = torch.zeros(n_bc_lower, 1)
    tau_bc_lower = torch.rand(n_bc_lower, 1) * T_max

    # Upper BC: V(S_max, tau) ≈ S_max - K*exp(-r*tau)
    S_bc_upper = torch.full((n_bc_upper, 1), S_max)
    tau_bc_upper = torch.rand(n_bc_upper, 1) * T_max

    return {
        'S_interior': S_interior, 'tau_interior': tau_interior,
        'S_ic': S_ic, 'tau_ic': tau_ic,
        'S_bc_lower': S_bc_lower, 'tau_bc_lower': tau_bc_lower,
        'S_bc_upper': S_bc_upper, 'tau_bc_upper': tau_bc_upper,
    }

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
data = generate_collocation(S_max, T_max, K, r, N_INTERIOR, N_IC, N_BC)

df_all = pd.concat([
    pd.DataFrame({
        'S': data['S_interior'].numpy().flatten(),
        'tau': data['tau_interior'].numpy().flatten(),
        'point_type': 'interior'
    }),
    pd.DataFrame({
        'S': data['S_ic'].numpy().flatten(),
        'tau': data['tau_ic'].numpy().flatten(),
        'point_type': 'initial_condition'
    }),
    pd.DataFrame({
        'S': data['S_bc_lower'].numpy().flatten(),
        'tau': data['tau_bc_lower'].numpy().flatten(),
        'point_type': 'boundary_condition'
    }),
    pd.DataFrame({
        'S': data['S_bc_upper'].numpy().flatten(),
        'tau': data['tau_bc_upper'].numpy().flatten(),
        'point_type': 'upper_boundary'
    }),
], ignore_index=True)

df_all.to_parquet(out_path)
print(f"Saved to {out_path}")
print(df_all['point_type'].value_counts())
print(f"\nS range: [{df_all['S'].min():.2f}, {df_all['S'].max():.2f}]")
print(f"tau range: [{df_all['tau'].min():.2f}, {df_all['tau'].max():.2f}]")